In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import re
import string

In [3]:
corpus = [
    "Natural language processing is a fascinating field",
    "Bag of words is a simple text representation technique",
    "TF-IDF improves on bag of words by weighting terms",
    "Machine learning models need numerical input like word vectors"
]

for i, doc in enumerate(corpus):
    print(f"Doc {i+1}: {doc}")

Doc 1: Natural language processing is a fascinating field
Doc 2: Bag of words is a simple text representation technique
Doc 3: TF-IDF improves on bag of words by weighting terms
Doc 4: Machine learning models need numerical input like word vectors


In [5]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\d+', '', text)
    text = text.strip()
    return text

cleaned_corpus = [preprocess(doc) for doc in corpus]
cleaned_corpus

['natural language processing is a fascinating field',
 'bag of words is a simple text representation technique',
 'tfidf improves on bag of words by weighting terms',
 'machine learning models need numerical input like word vectors']

In [9]:
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(cleaned_corpus)

bow_df = pd.DataFrame(bow_matrix.toarray(), columns=bow_vectorizer.get_feature_names_out())
bow_df.index = [f"Doc{i+1}" for i in range(len(corpus))]
bow_df


,bag,by,fascinating,field,improves,input,is,language,learning,like,...,representation,simple,technique,terms,text,tfidf,vectors,weighting,word,words
Doc1,0,0,1,1,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
Doc2,1,0,0,0,0,0,1,0,0,0,...,1,1,1,0,1,0,0,0,0,1
Doc3,1,1,0,0,1,0,0,0,0,0,...,0,0,0,1,0,1,0,1,0,1
Doc4,0,0,0,0,0,1,0,0,1,1,...,0,0,0,0,0,0,1,0,1,0


In [11]:
def build_vocab(docs):
    vocab = set()
    for doc in docs:
        vocab.update(doc.split())
    return sorted(vocab)

def manual_bow(docs):
    vocab = build_vocab(docs)
    vectors = []
    for doc in docs:
        words = doc.split()
        vector = [words.count(word) for word in vocab]
        vectors.append(vector)
    return pd.DataFrame(vectors, columns=vocab, index=[f"Doc{i+1}" for i in range(len(docs))])

manual_bow_df = manual_bow(cleaned_corpus)
manual_bow_df

,a,bag,by,fascinating,field,improves,input,is,language,learning,...,representation,simple,technique,terms,text,tfidf,vectors,weighting,word,words
Doc1,1,0,0,1,1,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
Doc2,1,1,0,0,0,0,0,1,0,0,...,1,1,1,0,1,0,0,0,0,1
Doc3,0,1,1,0,0,1,0,0,0,0,...,0,0,0,1,0,1,0,1,0,1
Doc4,0,0,0,0,0,0,1,0,0,1,...,0,0,0,0,0,0,1,0,1,0


In [13]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(cleaned_corpus)

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
tfidf_df.index = [f"Doc{i+1}" for i in range(len(corpus))]
tfidf_df.round(3)

,bag,by,fascinating,field,improves,input,is,language,learning,like,...,representation,simple,technique,terms,text,tfidf,vectors,weighting,word,words
Doc1,0.000,0.000,0.422,0.422,0.000,0.000,0.333,0.422,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Doc2,0.310,0.000,0.000,0.000,0.000,0.000,0.310,0.000,0.000,0.000,...,0.393,0.393,0.393,0.000,0.393,0.000,0.000,0.000,0.000,0.310
Doc3,0.281,0.357,0.000,0.000,0.357,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.357,0.000,0.357,0.000,0.357,0.000,0.281
Doc4,0.000,0.000,0.000,0.000,0.000,0.333,0.000,0.000,0.333,0.333,...,0.000,0.000,0.000,0.000,0.000,0.000,0.333,0.000,0.333,0.000


In [15]:
import math

def compute_tf(doc, vocab):
    words = doc.split()
    tf = {}
    for word in vocab:
        tf[word] = words.count(word) / len(words) if len(words) > 0 else 0
    return tf

def compute_idf(docs, vocab):
    N = len(docs)
    idf = {}
    for word in vocab:
        containing = sum(1 for doc in docs if word in doc.split())
        idf[word] = math.log(N / (1 + containing)) + 1
    return idf

vocab = build_vocab(cleaned_corpus)
idf_values = compute_idf(cleaned_corpus, vocab)

tfidf_manual = []
for doc in cleaned_corpus:
    tf_values = compute_tf(doc, vocab)
    row = [tf_values[word] * idf_values[word] for word in vocab]
    tfidf_manual.append(row)

manual_tfidf_df = pd.DataFrame(tfidf_manual, columns=vocab, index=[f"Doc{i+1}" for i in range(len(corpus))])
manual_tfidf_df.round(3)

,a,bag,by,fascinating,field,improves,input,is,language,learning,...,representation,simple,technique,terms,text,tfidf,vectors,weighting,word,words
Doc1,0.184,0.000,0.000,0.242,0.242,0.000,0.000,0.184,0.242,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Doc2,0.143,0.143,0.000,0.000,0.000,0.000,0.000,0.143,0.000,0.000,...,0.188,0.188,0.188,0.000,0.188,0.000,0.000,0.000,0.000,0.143
Doc3,0.000,0.143,0.188,0.000,0.000,0.188,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.188,0.000,0.188,0.000,0.188,0.000,0.143
Doc4,0.000,0.000,0.000,0.000,0.000,0.000,0.188,0.000,0.000,0.188,...,0.000,0.000,0.000,0.000,0.000,0.000,0.188,0.000,0.188,0.000


In [17]:
def get_top_keywords(tfidf_df, top_n=3):
    for doc in tfidf_df.index:
        top_words = tfidf_df.loc[doc].sort_values(ascending=False).head(top_n)
        print(f"\n{doc} top keywords:")
        print(top_words)

get_top_keywords(tfidf_df)


Doc1 top keywords:
processing     0.421765
fascinating    0.421765
field          0.421765
Name: Doc1, dtype: float64

Doc2 top keywords:
text         0.392644
technique    0.392644
simple       0.392644
Name: Doc2, dtype: float64

Doc3 top keywords:
by           0.35658
weighting    0.35658
tfidf        0.35658
Name: Doc3, dtype: float64

Doc4 top keywords:
numerical    0.333333
like         0.333333
need         0.333333
Name: Doc4, dtype: float64


In [19]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)
similarity_df = pd.DataFrame(similarity_matrix,
                              index=[f"Doc{i+1}" for i in range(len(corpus))],
                              columns=[f"Doc{i+1}" for i in range(len(corpus))])
similarity_df.round(3)

,Doc1,Doc2,Doc3,Doc4
Doc1,1.000,0.103,0.000,0.0
Doc2,0.103,1.000,0.261,0.0
Doc3,0.000,0.261,1.000,0.0
Doc4,0.000,0.000,0.000,1.0
